# BirdCLEF+ 2025 - Pretrained models, A1: inference (submission notebook)

This is a targeted copy of our team's baseline `infer-v2` notebook and uses the same protocol: **CPU only, internet off**, twelve 5 s windows per 1-minute soundscape. The model name, mel settings and normalisation come from the A1 training notebook's `cfg.json`, and the windows use the same `to_input` as training.

The hidden test set only appears when Kaggle scores a submission. In a normal run `test_soundscapes/` is empty, so the notebook does a dry run on a few unlabelled training soundscapes. Timing is logged per stage (decode, mel, model).


In [ ]:
import os, glob, json, time
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
import timm

def find_data_dir():
    if os.environ.get("M3_DATA_DIR"):       # local tests only
        return Path(os.environ["M3_DATA_DIR"])
    hits = glob.glob("/kaggle/input/**/birdclef-2025/train.csv", recursive=True) \
        or glob.glob("/kaggle/input/*/train.csv")
    return Path(hits[0]).parent if hits else Path("../../data")

def find_model_dir():
    if os.environ.get("M3_MODEL_DIR"):      # local tests only
        return Path(os.environ["M3_MODEL_DIR"])
    hits = glob.glob("/kaggle/input/**/model.pth", recursive=True)
    return Path(hits[0]).parent if hits else Path("../../outputs/train_baseline")

DATA, MODEL_DIR = find_data_dir(), find_model_dir()
CFG = json.load(open(MODEL_DIR / "cfg.json"))
species = json.load(open(MODEL_DIR / "species.json"))
CFG.setdefault("norm", "v2"); CFG.setdefault("norm_top_db", 80.0)   # M1 cfg.json has neither
NUM_WORKERS = int(os.environ.get("M3_INFER_WORKERS", os.cpu_count()))   # 0 = no Pool (Windows tests)
torch.set_num_threads(os.cpu_count())
print("model:", CFG["model"], "| norm:", CFG["norm"], "| timm", timm.__version__)
print("Data:", DATA, "| model:", MODEL_DIR, "| CPUs:", os.cpu_count())

sample_sub = pd.read_csv(DATA / "sample_submission.csv")
assert list(sample_sub.columns[1:]) == species, "species order differs from sample_submission"
print(sample_sub.iloc[:3, :4])

In [ ]:
test_files = sorted((DATA / "test_soundscapes").glob("*.ogg"))
DRY_RUN = len(test_files) == 0
if DRY_RUN:
    test_files = sorted((DATA / "train_soundscapes").glob("*.ogg"))[:20]
print(f"{'DRY RUN on' if DRY_RUN else 'Predicting'} {len(test_files)} soundscapes")

## Mel spectrograms (identical settings to training, read from `cfg.json`)

In [ ]:
WIN = CFG["sr"] * CFG["duration"]

TOP_DB = float(CFG["norm_top_db"])

def to_input(db):
    """Shared with the other A1 notebook, keep both copies identical.
    Floor at norm_top_db below this window's loudest point, then either M1's scaling ("v2")
    or the 2nd-place per-window z-score followed by min-max to [0, 1] ("zscore_minmax")."""
    db = np.maximum(db - db.max(), -TOP_DB)
    if CFG["norm"] == "zscore_minmax":
        x = (db - db.mean()) / (db.std() + 1e-6)
        return ((x - x.min()) / (x.max() - x.min() + 1e-6)).astype(np.float32)
    return ((db + TOP_DB) / TOP_DB).astype(np.float32)

def to_mel(y):
    m = librosa.feature.melspectrogram(
        y=y, sr=CFG["sr"], n_fft=CFG["n_fft"], hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"], fmin=CFG["fmin"], fmax=CFG["fmax"])
    return to_input(librosa.power_to_db(m, ref=np.max, top_db=TOP_DB))

def soundscape_windows(path):
    t0 = time.perf_counter()
    y, _ = sf.read(path, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    n_win = max(1, int(np.ceil(len(y) / WIN)))
    y = np.pad(y, (0, n_win * WIN - len(y)))
    t1 = time.perf_counter()
    mels = np.stack([to_mel(y[i * WIN:(i + 1) * WIN]) for i in range(n_win)])
    # row_id = <file stem>_<end time in seconds>, e.g. soundscape_8358733_5
    row_ids = [f"{path.stem}_{(i + 1) * CFG['duration']}" for i in range(n_win)]
    return row_ids, mels, t1 - t0, time.perf_counter() - t1

t0 = time.time()
if NUM_WORKERS > 0:
    with Pool(NUM_WORKERS) as pool:
        results = pool.map(soundscape_windows, test_files)
else:
    results = [soundscape_windows(p) for p in test_files]
t_wall_features = time.time() - t0
row_ids = [r for ids, *_ in results for r in ids]
X = np.concatenate([r[1] for r in results])
t_decode = sum(r[2] for r in results); t_mel = sum(r[3] for r in results)
print("Windows:", X.shape, f"in {time.time() - t0:.0f}s")

## Predict

In [ ]:
model = timm.create_model(CFG["model"], pretrained=False, in_chans=1, num_classes=len(species))
model.load_state_dict(torch.load(MODEL_DIR / "model.pth", map_location="cpu"))
model.eval()

t0 = time.time(); preds = []
with torch.inference_mode():
    for i in range(0, len(X), 64):
        xb = torch.from_numpy(X[i:i + 64])[:, None]
        preds.append(torch.sigmoid(model(xb)).numpy())
preds = np.concatenate(preds)
print("Predictions:", preds.shape, f"in {time.time() - t0:.0f}s")
t_model = time.time() - t0
print(f"TIMING | decode {t_decode:.1f}s and mel {t_mel:.1f}s (summed over workers) | "
      f"features wall {t_wall_features:.1f}s | model wall {t_model:.1f}s | "
      f"total wall {t_wall_features + t_model:.1f}s | {len(test_files)} files")

In [ ]:
sub = pd.DataFrame(preds, columns=species)
sub.insert(0, "row_id", row_ids)
sub.to_csv("submission.csv", index=False)
print(sub.shape)
sub.iloc[:5, :6]